In [1]:
import langchain
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [6]:
##simple llm call
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage
model=init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022BEDD52660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022BEE48D6A0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
messages=[
   SystemMessage(content="You are a helpful assistant"),#instruction to llm that how it needs to behave
   HumanMessage(content="What is the machine learning and AI")##input to the model
]

In [11]:
#invoking the model for getting the ai message(output)
response=model.invoke(messages)
response

AIMessage(content='**Machine Learning (ML) and Artificial Intelligence (AI)**\n\nMachine Learning and Artificial Intelligence are two closely related fields that have revolutionized the way we live, work, and interact with technology.\n\n**Artificial Intelligence (AI)**\n\nArtificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:\n\n1. **Reasoning**: Drawing inferences and making decisions based on data.\n2. **Problem-solving**: Finding solutions to complex problems.\n3. **Learning**: Improving performance on a task over time.\n4. **Perception**: Interpreting and understanding data from sensors or other sources.\n5. **Natural Language Processing**: Understanding and generating human language.\n\nAI systems can be categorized into two types:\n\n1. **Narrow or Weak AI**: Designed to perform a specific task, such as facial recognition or language translation.\n2. **General or Strong AI**: A hypothetical

In [12]:
print(response.content)

**Machine Learning (ML) and Artificial Intelligence (AI)**

Machine Learning and Artificial Intelligence are two closely related fields that have revolutionized the way we live, work, and interact with technology.

**Artificial Intelligence (AI)**

Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Reasoning**: Drawing inferences and making decisions based on data.
2. **Problem-solving**: Finding solutions to complex problems.
3. **Learning**: Improving performance on a task over time.
4. **Perception**: Interpreting and understanding data from sensors or other sources.
5. **Natural Language Processing**: Understanding and generating human language.

AI systems can be categorized into two types:

1. **Narrow or Weak AI**: Designed to perform a specific task, such as facial recognition or language translation.
2. **General or Strong AI**: A hypothetical AI system that possesses the abilit

In [13]:
for chunk in model.stream(messages):
    print(chunk.content,end="",flush=True)

**Machine Learning (ML) and Artificial Intelligence (AI): An Overview**

Machine Learning and Artificial Intelligence are two interconnected fields that have revolutionized the way we live, work, and interact with technology. While often used interchangeably, they have distinct meanings and applications.

**Artificial Intelligence (AI):**

Artificial Intelligence refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Reasoning**: Drawing inferences, making decisions, and solving problems.
2. **Learning**: Adapting to new data, experiences, or environments.
3. **Perception**: Interpreting and understanding sensory data, such as images, speech, or text.
4. **Natural Language Processing (NLP)**: Understanding, generating, and processing human language.

AI systems can be categorized into two types:

1. **Narrow or Weak AI**: Designed to perform a specific task, such as facial recognition, language translation, or play

In [15]:
##Dynamic Prompt Templates
from langchain_core.prompts import ChatPromptTemplate
translation_template=ChatPromptTemplate.from_messages([
    ("system","You are professional language translator. Translate the following {text} from {source_language} to {target_language}. Maintain the style and the tone of the original text."),
    ("user","{text}")
])
##using the template
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"French",
    "text":"I am learning langchain and it is very interesting"
})

In [18]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are professional language translator. Translate the following I am learning langchain and it is very interesting from English to French. Maintain the style and the tone of the original text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I am learning langchain and it is very interesting', additional_kwargs={}, response_metadata={})])

In [19]:
translated_response=model.invoke(prompt)
translated_response.content

"Je suis en train d'apprendre langchain et c'est très intéressant."

In [28]:
##BUilding chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
def create_story_chain():
    ##template for story generation
    story_prompt=ChatPromptTemplate.from_messages([
      ("system","You are a creative storyteller and you need to create a story based on the given {theme} and {genre}. The story should be engaging and should have a clear beginning, middle and end. The story should be around 500 words."),
      ("user","Theme: {theme}\nGenre: {genre}")
    ])

    ##template for story analysis
    analysis_prompt=ChatPromptTemplate.from_messages([
        ("system","You are a story analyst and you need to analyze the story based on the given {story}. The analysis should include the main theme, the characters, the plot and the overall message of the story."),
        ("user","{story}")
    ])
    ##strouput pareser gives the content directly and not the list of other faltu things
    story_chain=(
        story_prompt| model| StrOutputParser()
    )
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain=(
        story_chain| RunnableLambda(analyze_story) |analysis_prompt| model| StrOutputParser()
    )
    return analysis_chain

In [29]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['genre', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre', 'theme'], input_types={}, partial_variables={}, template='You are a creative storyteller and you need to create a story based on the given {theme} and {genre}. The story should be engaging and should have a clear beginning, middle and end. The story should be around 500 words.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nGenre: {genre}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object 

In [30]:
result=chain.invoke({
    "theme":"Overcoming adversity",
    "genre":"Drama"
})
print("Generated Story and Analysis:\n")
print(result)

Generated Story and Analysis:

**Story Analysis: The Unbreakable Spirit**

**Main Theme:**
The main theme of the story is the power of resilience and community in overcoming adversity. The story highlights how a small community, led by a determined individual, can come together to overcome even the most daunting challenges. The theme is reinforced by the transformation of the neglected community garden into a thriving oasis, which serves as a symbol of hope and renewal.

**Characters:**

* **Sarah:** The protagonist of the story, Sarah is a young woman who embodies the unbreakable spirit. She is determined, resourceful, and courageous in the face of adversity. Her love for her family and community drives her to take action and inspire others to do the same.
* **John:** Sarah's husband, John, is a supportive character who helps Sarah in her mission to transform the garden. His loss of job and subsequent involvement in the garden project serves as a reminder of the economic struggles fac